# CTA — Gate 2 · Level 1: LSE-pooling in KV space

**The black hole we are closing.** The original Gate 2 collapsed the *embeddings* of a repeated span into one mean vector `e_bar = Σ πᵢ xᵢ`, then let the model re-project it into a single key/value. Result on Qwen2.5-3B: prefill 8.3× faster, KV 3.9× lighter, but **PPL 1.67 → 3.20** — the model kept the topic but lost specifics ('OAuth2, TTL 3600s' → 'username/password, time'). 

**Root cause (math).** Attention is non-linear in the query `q`:

$$o_{true}(q)=\sum_i \frac{e^{q\cdot k_i}}{\sum_j e^{q\cdot k_j}}\,v_i$$

The softmax weights depend on `q`, which is unknown at collapse time. A mean of embeddings, projected once, cannot carry the span's **attention mass** — it is missing the `log k` term. So it under-weights the whole span among the rest of the context.

**Level 1 fix.** Collapse **after** the K/V projection, directly on the cached keys/values, with log-sum-exp:

$$q\cdot \bar k \;\overset{!}{=}\; \log\sum_i e^{q\cdot k_i}\;\approx\; \log k + q\cdot\mu_k + \tfrac12 q^\top\Sigma_k q$$
$$\Rightarrow\quad \bar k=\mu_k+\frac{\log k}{\lVert q^\star\rVert^2}q^\star+\tfrac12\Sigma_k q^\star,\qquad \bar v=\sum_i \omega_i v_i,\ \ \omega_i=\mathrm{softmax}\big(\lVert k_i\rVert/\sqrt d\big)$$

The `log k` **mass term** is exactly what CTA-mean lacks and is the main expected win; the quadratic term adds curvature from the key covariance `Σ_k`.

**What this notebook measures.** On the SAME RAG prompt, four decode paths — **baseline**, **CTA-mean**, **CTA-LSE (mass+quad)**, **CTA-LSE (mass-only)** — compared on PPL (honest judge = full model), greedy-match, decode ms/tok, KV size. Plus the `Σ_k` spectrum → a data-driven signal for whether Level 2 (rank-r) is needed.

Run top-to-bottom on a **T4**. ~3-4 min.

## 0 · Environment check

In [ ]:
import torch, sys
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))

## 1 · Install deps + fetch CTA source (includes `cta.kv_lse`)

In [ ]:
import os, subprocess
def sh(cmd):
    print('$', cmd); return subprocess.run(cmd, shell=True).returncode
sh('pip -q install -U transformers accelerate >/dev/null 2>&1')
if not os.path.isdir('compositional-token-algebra'):
    sh('git clone -q https://github.com/VadShv/compositional-token-algebra')
else:
    sh('cd compositional-token-algebra && git pull -q')
sys.path.insert(0, '/content/compositional-token-algebra/src')
print('ok')

## 2 · Load model (Qwen2.5-3B, bf16)
Same backbone as prefill + original Gate 2. Falls back to 0.5B if VRAM is tight.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME = 'Qwen/Qwen2.5-3B'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
try:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, attn_implementation='eager').to(DEVICE)
except Exception as e:
    print('3B failed, falling back to 0.5B:', str(e)[:100])
    MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, attn_implementation='eager').to(DEVICE)
model.eval()
cfg = model.config
print(f'model={MODEL_NAME} layers={cfg.num_hidden_layers} d={cfg.hidden_size} device={DEVICE} dtype={DTYPE}')

## 3 · Helpers + the four decode paths
- **A baseline** — full-prompt decode, KV length `n`.
- **B CTA-mean** — pool embeddings → prefill → KV length `m` → decode (original Gate 2).
- **C CTA-LSE (mass+quad)** — full prefill → LSE-compress the KV-cache → KV length `m` → decode.
- **D CTA-LSE (mass-only)** — same, quadratic curvature term off (ablation).

Judge = `ppl_of_continuation` (perplexity under the FULL uncollapsed model).

In [ ]:
import time, json, random
from cta.detector import select_collapse_spans, build_segments
from cta.algebra import compose
from cta.kv_lse import collapse_cache_lse, build_dynamic_cache, sigma_spectrum
N_GEN = 64

def emb_of(ids):
    return model.model.embed_tokens(ids)
def seg_of(ids):
    return build_segments(ids.tolist(), select_collapse_spans(ids.tolist(), k_min=3, k_max=16, f_min=2))

@torch.no_grad()
def decode_over_cache(past, first_hidden, start_pos, n_gen):
    nid = int(model.lm_head(first_hidden).argmax(-1)); gen=[nid]; cur=start_pos
    if DEVICE=='cuda': torch.cuda.synchronize()
    t0=time.time()
    for _ in range(n_gen-1):
        te = model.model.embed_tokens(torch.tensor([[nid]], device=DEVICE)).to(DTYPE)
        pid = torch.tensor([[cur]], device=DEVICE)
        out = model.model(inputs_embeds=te, past_key_values=past, use_cache=True, position_ids=pid)
        past = out.past_key_values
        nid = int(model.lm_head(out.last_hidden_state[:, -1, :]).argmax(-1))
        gen.append(nid); cur+=1
    if DEVICE=='cuda': torch.cuda.synchronize()
    return gen, (time.time()-t0)/max(1,n_gen-1)*1000

@torch.no_grad()
def path_baseline(ids, n_gen):
    emb = emb_of(ids).unsqueeze(0)
    if DEVICE=='cuda': torch.cuda.synchronize()
    t0=time.time(); out = model.model(inputs_embeds=emb, use_cache=True)
    if DEVICE=='cuda': torch.cuda.synchronize()
    pre=(time.time()-t0)*1000; past=out.past_key_values; L=ids.shape[0]
    gen,per = decode_over_cache(past, out.last_hidden_state[:,-1,:], L, n_gen)
    return gen, pre, per, past.get_seq_length()

@torch.no_grad()
def path_cta_mean(ids, n_gen, mode='norm'):
    emb = emb_of(ids); segs = seg_of(ids); vecs=[]
    for (s,e,is_col) in segs:
        vecs.append(compose(emb[s:e], mode=mode)[0] if is_col else emb[s])
    collapsed = torch.stack(vecs,0).unsqueeze(0)
    if DEVICE=='cuda': torch.cuda.synchronize()
    t0=time.time(); out = model.model(inputs_embeds=collapsed, use_cache=True)
    if DEVICE=='cuda': torch.cuda.synchronize()
    pre=(time.time()-t0)*1000; past=out.past_key_values; m=collapsed.shape[1]
    gen,per = decode_over_cache(past, out.last_hidden_state[:,-1,:], m, n_gen)
    return gen, pre, per, past.get_seq_length()

@torch.no_grad()
def path_cta_lse(ids, n_gen, use_quad=True, mass=True):
    emb = emb_of(ids).unsqueeze(0); segs = seg_of(ids)
    if DEVICE=='cuda': torch.cuda.synchronize()
    t0=time.time(); out = model.model(inputs_embeds=emb, use_cache=True)
    full_past = out.past_key_values
    new_layers = collapse_cache_lse(full_past, segs, q_ref_per_layer=None, use_quad=use_quad, mass=mass)
    comp = build_dynamic_cache(new_layers)
    if DEVICE=='cuda': torch.cuda.synchronize()
    pre=(time.time()-t0)*1000; m=comp.get_seq_length()
    gen,per = decode_over_cache(comp, out.last_hidden_state[:,-1,:], m, n_gen)
    spec=None
    try:
        K0 = full_past.layers[0].keys[0] if hasattr(full_past,'layers') else full_past.key_cache[0][0]
        for (s,e,is_col) in segs:
            if is_col and (e-s)>=2:
                spec = sigma_spectrum(K0[:,s:e,:]); break
    except Exception: spec=None
    return gen, pre, per, m, spec

@torch.no_grad()
def ppl_of_continuation(prompt_ids, gen_ids):
    full = torch.cat([prompt_ids, torch.tensor(gen_ids, device=DEVICE)])
    out = model.model(inputs_embeds=emb_of(full).unsqueeze(0))
    logits = model.lm_head(out.last_hidden_state)[0]
    n=len(prompt_ids); tgt=torch.tensor(gen_ids, device=DEVICE)
    lp = torch.log_softmax(logits[n-1:n-1+len(gen_ids)].float(), -1)
    ll = lp[torch.arange(len(gen_ids)), tgt]
    return float(torch.exp(-ll.mean()))

def match_rate(a,b): return round(sum(1 for x,y in zip(a,b) if x==y)/len(a),3)
print('helpers ready')

## 4 · RAG prompt + run all four paths
Same synthetic RAG prompt as the original Gate 2 (10 near-duplicate retrieved docs → summarize).

In [ ]:
def make_rag_prompt():
    doc=('Retrieved document. User {u} logged in from 10.0.{a}.{b} at 2026-07-{d}. '
         'Session established. Role=admin, region=eu-west. Auth method oauth2. '
         'Token TTL 3600s. Rate limit 1000 req/min. Endpoint /api/v2/login returned 200 OK. ')
    random.seed(0); ctx=''
    for i in range(10):
        ctx += doc.format(u=1000+i, a=random.randint(0,255), b=random.randint(0,255), d=random.randint(10,28))
    return ctx + ('\n\nBased on the retrieved documents above, summarize the common '
                  'authentication configuration in one sentence:')

prompt = make_rag_prompt()
ids = tok(prompt, add_special_tokens=False, return_tensors='pt').input_ids[0].to(DEVICE)
n = len(ids)
a_ids,a_pre,a_per,a_kv = path_baseline(ids, N_GEN)
b_ids,b_pre,b_per,b_kv = path_cta_mean(ids, N_GEN)
c_ids,c_pre,c_per,c_m,c_spec = path_cta_lse(ids, N_GEN, use_quad=True, mass=True)
d_ids,d_pre,d_per,d_m,_ = path_cta_lse(ids, N_GEN, use_quad=False, mass=True)
print(f'n={n}  m={b_kv}  collapse_ratio={b_kv/n:.3f}')

## 5 · Metrics table + generated text

In [ ]:
res = dict(model=MODEL_NAME, n_prompt=n, n_gen=N_GEN, collapse_ratio=round(b_kv/n,4),
  baseline=dict(prefill_ms=round(a_pre,1), per_tok_ms=round(a_per,2), kv_positions=a_kv,
                ppl_self=round(ppl_of_continuation(ids,a_ids),3)),
  cta_mean=dict(prefill_ms=round(b_pre,1), per_tok_ms=round(b_per,2), kv_positions=b_kv,
                ppl=round(ppl_of_continuation(ids,b_ids),3), greedy_match=match_rate(a_ids,b_ids)),
  cta_lse_mass_quad=dict(prefill_ms=round(c_pre,1), per_tok_ms=round(c_per,2), kv_positions=c_m,
                ppl=round(ppl_of_continuation(ids,c_ids),3), greedy_match=match_rate(a_ids,c_ids)),
  cta_lse_mass_only=dict(prefill_ms=round(d_pre,1), per_tok_ms=round(d_per,2), kv_positions=d_m,
                ppl=round(ppl_of_continuation(ids,d_ids),3), greedy_match=match_rate(a_ids,d_ids)),
  sigma_k_top5_layer0_span0=c_spec,
  baseline_text=tok.decode(a_ids), cta_mean_text=tok.decode(b_ids),
  cta_lse_text=tok.decode(c_ids))
print(json.dumps({k:v for k,v in res.items() if not k.endswith('_text')}, indent=2))
print('\n--- BASELINE ---\n', res['baseline_text'])
print('\n--- CTA-MEAN (old) ---\n', res['cta_mean_text'])
print('\n--- CTA-LSE (mass+quad, NEW) ---\n', res['cta_lse_text'])

## 6 · Save → send `results_gate2_lse.json` back

In [ ]:
with open('results_gate2_lse.json','w') as f:
    json.dump(res, f, indent=2, ensure_ascii=False)
print('saved results_gate2_lse.json')
try:
    from google.colab import files; files.download('results_gate2_lse.json')
except Exception: pass

## How to read the result

The KV win is identical for all CTA paths (cache length `m`), so this is purely a **quality contest**:

- **GO for Level 1** — `cta_lse_*` PPL moves meaningfully back toward baseline vs `cta_mean` (e.g. 3.20 → <2.5), text keeps more specifics. The `log k` mass term did its job → ship LSE-pooling as the default collapse.
- **PARTIAL** — LSE helps but a gap remains, and `sigma_k_top5` shows 2-3 large eigenvalues → key dispersion is real; escalate to **Level 2 (rank-r)** where the attention error is bounded by `σ_{r+1}`.
- **NO-GO** — LSE ≈ mean or worse → the loss is not in the key/value aggregation but elsewhere (position/RoPE or the value pooling); revisit `q*` choice or go straight to Level 3 (lazy-expand).

`mass_quad` vs `mass_only` isolates whether the quadratic curvature term is worth its cost. `sigma_k_top5` is the escalation signal: a fast-decaying spectrum ⇒ one vector suffices (Level 1 enough); a flat spectrum ⇒ need rank-r.